# Build a QtlDataset per study

## Description

Construct a single pecotmr `QtlDataset` from one study's genotype + per-context phenotype + per-context covariate files (plus an optional uniform genotype-PC file), and serialize it to a single RDS. The resulting object is the upstream dependency for every per-gene fineMapping / TWAS / colocboost task — gene-level parallelization happens against this single object via per-gene `traitId` selectors inside the downstream pipelines.

This step does NOT iterate over genes or regions. Build the QtlDataset once per study; downstream notebooks fan out per-gene tasks against the resulting RDS.

## Inputs

- `--study` &mdash; study identifier.
- `--genotype-prefix` &mdash; PLINK1 bed/bim/fam prefix (no extension).
- `--phenotype-manifest` &mdash; TSV with columns `#chr, start, end, ID, path, cond` (and optionally `cov_path`). One row per (region, context). Relative `path` / `cov_path` entries resolve against the manifest's own directory.
- `--genotype-covariates` &mdash; optional TSV of genotype-derived covariates (e.g. ancestry PCs) applied uniformly across all contexts.
- `--transpose-covariates` &mdash; when set, transposes every covariate TSV (phenotype + genotype) after reading. Use this for QTLtools-format inputs where rows are covariate names and columns are samples.
- `--maf-cutoff` / `--xvar-cutoff` &mdash; pass-through to `QtlDataset()`. Lazy filters applied inside the accessors at fit time.

## Output

- `{cwd}/{study}.qtl_dataset.rds`


## Example

```bash
sos run pipeline/qtl_dataset.ipynb qtl_dataset_construct \
    --cwd output \
    --study TEST_STUDY \
    --genotype-prefix input/colocboost/example.chr22 \
    --phenotype-manifest input/colocboost/pheno_manifest_multicontext.tsv \
    --transpose-covariates
```


In [ ]:
[global]
parameter: cwd = path('output')
parameter: study = str
parameter: genotype_prefix = path
parameter: phenotype_manifest = path
parameter: genotype_covariates = path('.')
# Set when covariate TSVs are in QTLtools format (rows = covariates, cols = samples).
parameter: transpose_covariates = False
parameter: maf_cutoff = 0.0
parameter: xvar_cutoff = 0.0
# Directory holding code/script of the xqtl-protocol checkout; override
# when SoS is invoked from a working dir that doesn't contain the scripts.
parameter: modular_script_dir = path('code/script')
parameter: container = ''
parameter: walltime = '15m'
parameter: mem = '8G'
parameter: numThreads = 1


In [ ]:
[qtl_dataset_construct]
# Build the QtlDataset once. No fan-out; downstream notebooks load this
# RDS and parallelize per gene.
output: f"{cwd}/{study}.qtl_dataset.rds"
task: trunk_workers = 1, trunk_size = 1, walltime = walltime, mem = mem, cores = numThreads, tags = f"{step_name}_{_output:bn}"
bash: expand = '${ }', stderr = f"{_output}.stderr", stdout = f"{_output}.stdout", container = container
    Rscript ${modular_script_dir}/pecotmr_integration/qtl_dataset_construct.R \
        --study ${study} \
        --genotype-prefix ${genotype_prefix} \
        --phenotype-manifest ${phenotype_manifest} \
        --genotype-covariates ${genotype_covariates if genotype_covariates.is_file() else '""'} \
        ${'--transpose-covariates' if transpose_covariates else ''} \
        --maf-cutoff ${maf_cutoff} \
        --xvar-cutoff ${xvar_cutoff} \
        --output ${_output}
